In [7]:
import pdb
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
#from .utils import *
import pdb
import matplotlib.pyplot as plt
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from timm.models.layers import DropPath, trunc_normal_
from timm.models.registry import register_model
#from timm.models.layers.helpers import to_2tuple
from timm.layers.helpers import to_2tuple
#from pyheatmap.heatmap import HeatMap
import random
#from pyheatmap.heatmap import HeatMap
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

In [5]:
#!pip install timm

In [8]:
#from .utils import *
import torch.nn as nn


class qkv_transform(nn.Conv1d):
    """Conv1d for qkv_transform"""

In [9]:
def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv3d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


In [10]:
class GroupNorm(nn.GroupNorm):
    """
    Group Normalization with 1 group.
    Input: tensor in shape [B, C, H, W]
    """
    def __init__(self, num_channels, **kwargs):
        super().__init__(1, num_channels, **kwargs)


In [12]:
class AxialAttention_dynamic(nn.Module):
    def __init__(self, in_planes, out_planes, groups=8, kernel_size=56,
                 stride=1, bias=False, width=False, length = False):
        assert (in_planes % groups == 0) and (out_planes % groups == 0)
        super(AxialAttention_dynamic, self).__init__()
        self.in_planes = in_planes
        self.out_planes = out_planes
        self.groups = groups
        self.group_planes = out_planes // groups
        self.kernel_size = kernel_size
        self.stride = stride
        self.bias = bias
        self.width = width
        self.length = length
        # Multi-head self attention
        self.qkv_transform = qkv_transform(in_planes, out_planes * 2, kernel_size=1, stride=1,
                                           padding=0, bias=False)
        self.bn_qkv = nn.BatchNorm1d(out_planes * 2)
        self.bn_similarity = nn.BatchNorm2d(groups * 3)
        self.bn_output = nn.BatchNorm1d(out_planes * 2)

        # Priority on encoding

        ## Initial values

        self.f_qr = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_kr = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_sve = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_sv = nn.Parameter(torch.tensor(1.0),  requires_grad=False)


        # Position embedding
        self.relative = nn.Parameter(torch.randn(self.group_planes * 2, kernel_size * 2 - 1), requires_grad=True)
        query_index = torch.arange(kernel_size).unsqueeze(0)
        key_index = torch.arange(kernel_size).unsqueeze(1)
        relative_index = key_index - query_index + kernel_size - 1
        self.register_buffer('flatten_index', relative_index.view(-1))
        if stride > 1:
            self.pooling = nn.AvgPool3d(3, stride=(2,2,2),padding=1)

        self.reset_parameters()
        # self.print_para()

    def forward(self, x): # N, C, L, H, W
        if self.length:
            x = x.permute(0, 3, 4, 1, 2) # N, H, W, C, L
        else:
          if self.width:
            x = x.permute(0, 2, 3, 1, 4)  # N, L, H, C, W
          else:
            x = x.permute(0, 2, 4, 1, 3)  # N, L, W, C, H
        N, L, W, C, H = x.shape
        
        x = x.contiguous().view(N * W * L, C, H)

        # Transformations
        qkv = self.bn_qkv(self.qkv_transform(x))
        q, k, v = torch.split(qkv.reshape(N * L * W, self.groups, self.group_planes * 2, H), [self.group_planes // 2, self.group_planes // 2, self.group_planes], dim=2)

        # Calculate position embedding
        all_embeddings = torch.index_select(self.relative, 1, self.flatten_index).view(self.group_planes * 2, self.kernel_size, self.kernel_size)
        q_embedding, k_embedding, v_embedding = torch.split(all_embeddings, [self.group_planes // 2, self.group_planes // 2, self.group_planes], dim=0)
        qr = torch.einsum('bgci,cij->bgij', q, q_embedding)
        kr = torch.einsum('bgci,cij->bgij', k, k_embedding).transpose(2, 3)
        qk = torch.einsum('bgci, bgcj->bgij', q, k)


        # multiply by factors
        qr = torch.mul(qr, self.f_qr)
        kr = torch.mul(kr, self.f_kr)

        stacked_similarity = torch.cat([qk, qr, kr], dim=1)
        stacked_similarity = self.bn_similarity(stacked_similarity).view(N * W * L, 3, self.groups, H, H).sum(dim=1)
        #stacked_similarity = self.bn_qr(qr) + self.bn_kr(kr) + self.bn_qk(qk)
        # (N, groups, H, H, W)
        similarity = F.softmax(stacked_similarity, dim=3)
        sv = torch.einsum('bgij,bgcj->bgci', similarity, v)
        sve = torch.einsum('bgij,cij->bgci', similarity, v_embedding)

        # multiply by factors
        sv = torch.mul(sv, self.f_sv)
        sve = torch.mul(sve, self.f_sve)

        stacked_output = torch.cat([sv, sve], dim=-1).view(N * L * W, self.out_planes * 2, H)
        output = self.bn_output(stacked_output).view(N, L, W, self.out_planes, 2, H).sum(dim=-2)

        if self.length:
            output = output.permute(0, 3, 4, 1, 2)
        else:
          if self.width:
            output = output.permute(0, 3, 1, 2, 4)  # N, W, C, H
          else:
            output = output.permute(0, 3, 1, 4, 2)
        
        if self.stride > 1:
            output = self.pooling(output)

        return output

        def reset_parameters(self):
            self.qkv_transform.weight.data.normal_(0, math.sqrt(1. / self.in_planes))
            #nn.init.uniform_(self.relative, -0.1, 0.1)
            nn.init.normal_(self.relative, 0., math.sqrt(1. / self.group_planes))

In [13]:
class AxialBlock_dynamic(nn.Module):
    expansion = 2

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None, kernel_size=56, length=16):
        super(AxialBlock_dynamic, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.))
        # Both self.conv2 and self.downsample layers downsample the input when stride != 1
        self.conv_down = conv1x1(inplanes, width)
        self.bn1 = norm_layer(4,width)
        self.hight_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size=kernel_size)
        self.width_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size=kernel_size, width=True)
        self.length_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size = length, stride=stride, length=True)
        self.conv_up = conv1x1(width, planes * self.expansion)
        self.bn2 = norm_layer(4,planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv_down(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.hight_block(out)
        out = self.width_block(out)
        out = self.length_block(out)
        
        out = self.relu(out)

        out = self.conv_up(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        out = self.relu(out)

        return out


In [14]:
class ResAxialAttentionUNet3D(nn.Module):
    # model = ResAxialAttentionUNet(AxialBlock_dynamic, [1, 2, 4, 1], s= 0.125, **kwargs)
    def __init__(self, block, layers, num_classes=4, zero_init_residual=True,
                 groups=8, width_per_group=64, replace_stride_with_dilation=None,
                 norm_layer=None, s=0.125, img_size=128, img_depth = 128,imgchan=3):
        super(ResAxialAttentionUNet3D, self).__init__()
        if norm_layer is None:
            norm_layer = nn.GroupNorm
        self._norm_layer = norm_layer

        self.inplanes = int(128 * s)
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError("replace_stride_with_dilation should be None "
                             "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv3d(imgchan, self.inplanes, kernel_size=7, stride=(2,2,2), padding=3,
                               bias=False)
        self.conv2 = nn.Conv3d(self.inplanes, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv3 = nn.Conv3d(128, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = norm_layer(4,self.inplanes)
        self.bn2 = norm_layer(4,128)
        self.bn3 = norm_layer(4,self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.AvgPool3d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, int(128 * s), layers[0], stride=1, kernel_size=(img_size // 4),length = img_depth//4)
        self.layer2 = self._make_layer(block, int(256 * s), layers[1], stride=2, kernel_size=(img_size // 4),
                                       dilate=replace_stride_with_dilation[0],length = img_depth//4)
        self.layer3 = self._make_layer(block, int(512 * s), layers[2], stride=2, kernel_size=(img_size // 8),
                                       dilate=replace_stride_with_dilation[1],length = img_depth//8)
        self.layer4 = self._make_layer(block, int(1024 * s), layers[3], stride=1, kernel_size=(img_size // 16),
                                       dilate=replace_stride_with_dilation[2],length = img_depth//16)
        #self.channel = conv1x1(int(64 * s),int(128 * s) , 1)
        # Decoder
        self.decoder1 = nn.Conv3d(int(1024 * 2 * s), int(1024 * 2 * s), kernel_size=3, stride=(2,2,2), padding=1)
        self.decoder2 = nn.Conv3d(int(1024  * s), int(1024 * s), kernel_size=3, stride=2, padding=1)
        self.decoder3 = nn.Conv3d(int(1024 * s), int(512 * s), kernel_size=3, stride=1, padding=1)
        self.decoder4 = nn.Conv3d(int(512 * s), int(256 * s), kernel_size=3, stride=1, padding=1)
        self.decoder5 = nn.Conv3d(int(256 * s), int(128 * s), kernel_size=3, stride=1, padding=1)
        self.decoder6 = nn.Conv3d(int(128 * s), int(64 * s), kernel_size=3, stride=1, padding=1)
        self.adjust = nn.Conv3d(int(64 * s), num_classes, kernel_size=1, stride=1, padding=0)
        

        self.soft = nn.Softmax(dim=1)

    def _make_layer(self, block, planes, blocks, kernel_size=56, stride=1, dilate=False, length=16):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(4,planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, groups=self.groups,
                            base_width=self.base_width, dilation=previous_dilation,
                            norm_layer=norm_layer, kernel_size=kernel_size, length = length))
        self.inplanes = planes * block.expansion
        if stride != 1:
            kernel_size = kernel_size // 2
            length = length // 2
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer, kernel_size=kernel_size, length = length))

        return nn.Sequential(*layers)

    def _forward_impl(self, x):

        # AxialAttention Encoder
        # pdb.set_trace()

        
      x_slice = self.conv1(x)
      x_slice = self.bn1(x_slice)
        
      x_slice = self.relu(x_slice)
      x_slice = self.conv2(x_slice)
      x_slice = self.bn2(x_slice)
      x_slice = self.relu(x_slice)
      x_slice = self.conv3(x_slice)
      x_slice = self.bn3(x_slice)
      x_slice = self.relu(x_slice)
      x_bfpool = x_slice
      x_slice = self.pool(x_slice)

      x1 = self.layer1(x_slice)

      x2 = self.layer2(x1)

      x3 = self.layer3(x2)
      #x4 = self.layer4(x3)

      #x_slice = F.relu(F.interpolate(self.decoder1(x4), scale_factor=(2, 2, 2), mode='nearest'))

      #x_slice = torch.add(x_slice, x4)
      x_slice = F.relu(F.interpolate(self.decoder2(x3), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x3)
      x_slice = F.relu(F.interpolate(self.decoder3(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x2)
      x_slice = F.relu(F.interpolate(self.decoder4(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x1)
      x_slice = F.relu(F.interpolate(self.decoder5(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      #print(x_slice.shape)
      #print(x_bfpool.shape)
      #x_bfpool = self.channel(x_bfpool)
      #x_slice = torch.add(x_slice, x_bfpool)
      x_slice = F.relu(F.interpolate(self.decoder6(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = self.adjust(F.relu(x_slice))
      # pdb.set_trace()
        
      
      return x_slice

    def forward(self, x):
        return self._forward_impl(x)


In [15]:
def gated_3d(pretrained=False, **kwargs):
    model = ResAxialAttentionUNet3D(AxialBlock_dynamic, [1, 2, 4, 1], s= 0.25, **kwargs)
    return model